In [1]:
import pandas as pd
from geopy.distance import geodesic
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


df_raw = pd.read_csv("csv_building_structure.csv")


cols_to_drop = [
    'count_floors_post_eq',
    'height_ft_post_eq',
    'condition_post_eq',
    'technical_solution_proposed'
    ]


df_final = df_raw.drop(columns= cols_to_drop)

In [2]:
df_geo = pd.read_csv("ward_level_pred_intensity.csv")

# 2. Force strings and strip hidden whitespaces to ensure clean matching
# (This avoids false positives caused by int vs. string type mismatches)
df_final['ward_id'] = df_final['ward_id'].astype(str).str.strip()
df_geo['ward_id'] = df_geo['ward_id'].astype(str).str.strip()

# 3. Get unique sets of Ward IDs
building_wards = set(df_final['ward_id'].unique())
lookup_wards = set(df_geo['ward_id'].unique())

# 4. Find wards that are in the buildings data but MISSING from the lookup table
missing_in_lookup = building_wards - lookup_wards

# 5. Output the results
print("=== WARD ID AUDIT REPORT ===")
print(f"Total unique wards in Building data: {len(building_wards):,}")
print(f"Total unique wards in Lookup data:   {len(lookup_wards):,}")
print(f"Number of missing wards in Lookup:   {len(missing_in_lookup):,}")
print("============================\n")

if len(missing_in_lookup) > 0:
    # Look at a sample of the missing IDs
    print("⚠️ Sample of missing Ward IDs from the lookup dataset:")
    print(list(missing_in_lookup)[:10])
    
    # Calculate how many building records are affected by these missing wards
    affected_rows = df_final['ward_id'].isin(missing_in_lookup).sum()
    pct_affected = (affected_rows / len(df_final)) * 100
    print(f"\nImpact: This affects {affected_rows:,} building records ({pct_affected:.2f}% of your data).")
else:
    print("✅ Success! Every single ward_id in your building dataset exists inside the lookup dataset.")

=== WARD ID AUDIT REPORT ===
Total unique wards in Building data: 945
Total unique wards in Lookup data:   946
Number of missing wards in Lookup:   2

⚠️ Sample of missing Ward IDs from the lookup dataset:
['230209', '200608']

Impact: This affects 2,096 building records (0.28% of your data).


In [3]:
df_final

,building_id,district_id,vdcmun_id,ward_id,count_floors_pre_eq,age_building,plinth_area_sq_ft,height_ft_pre_eq,land_surface_condition,foundation_type,...,has_superstructure_stone_flag,has_superstructure_cement_mortar_stone,has_superstructure_mud_mortar_brick,has_superstructure_cement_mortar_brick,has_superstructure_timber,has_superstructure_bamboo,has_superstructure_rc_non_engineered,has_superstructure_rc_engineered,has_superstructure_other,damage_grade
0,120101000011,12,1207,120703,1,9,288,9,Flat,Other,...,0,0,0,0,0,1,0,0,0,Grade 3
1,120101000021,12,1207,120703,1,15,364,9,Flat,Other,...,0,0,0,0,0,1,0,0,0,Grade 5
2,120101000031,12,1207,120703,1,20,384,9,Flat,Other,...,0,0,0,0,0,0,0,0,0,Grade 2
3,120101000041,12,1207,120703,1,20,312,9,Flat,Other,...,0,0,0,0,0,0,0,0,0,Grade 2
4,120101000051,12,1207,120703,1,30,308,9,Flat,Other,...,0,0,0,0,0,0,0,0,0,Grade 1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
762101,366709001241,36,3603,360302,2,60,165,18,Flat,Mud mortar-Stone/Brick,...,0,0,0,0,0,0,0,0,0,Grade 5
762102,366709001251,36,3603,360302,2,35,342,18,Flat,Mud mortar-Stone/Brick,...,0,0,0,0,0,0,0,0,0,Grade 5
762103,366709001261,36,3603,360302,2,35,342,18,Flat,Mud mortar-Stone/Brick,...,0,0,0,0,0,0,0,0,0,Grade 5
762104,366709001271,36,3603,360302,2,19,306,18,Flat,Mud mortar-Stone/Brick,...,0,0,0,0,0,0,0,0,0,Grade 5


In [4]:
df_geo['ward_id'] = df_geo['ward_id'].astype('int')
df_final['ward_id'] = df_final['ward_id'].astype('int')

# We use a left join to ensure we keep every single building record
df_merged = df_final.merge(df_geo[['ward_id','latitude', 'longitude']], on='ward_id', how='left')

# 2. Check for missing coordinates due to unmatched ward_ids
missing_coords = df_merged['latitude'].isna().sum()
if missing_coords > 0:
    print(f"⚠️ Warning: {missing_coords} buildings did not match a ward_id.")
    print("Rows before", len(df_merged))
    df_merged.dropna(axis=0, inplace= True)

print("Rows after", len(df_merged))
df_merged

⚠️ Warning: 2096 buildings did not match a ward_id.
Rows before 762106
Rows after 759998


,building_id,district_id,vdcmun_id,ward_id,count_floors_pre_eq,age_building,plinth_area_sq_ft,height_ft_pre_eq,land_surface_condition,foundation_type,...,has_superstructure_mud_mortar_brick,has_superstructure_cement_mortar_brick,has_superstructure_timber,has_superstructure_bamboo,has_superstructure_rc_non_engineered,has_superstructure_rc_engineered,has_superstructure_other,damage_grade,latitude,longitude
0,120101000011,12,1207,120703,1,9,288,9,Flat,Other,...,0,0,0,1,0,0,0,Grade 3,27.28525,86.526839
1,120101000021,12,1207,120703,1,15,364,9,Flat,Other,...,0,0,0,1,0,0,0,Grade 5,27.28525,86.526839
2,120101000031,12,1207,120703,1,20,384,9,Flat,Other,...,0,0,0,0,0,0,0,Grade 2,27.28525,86.526839
3,120101000041,12,1207,120703,1,20,312,9,Flat,Other,...,0,0,0,0,0,0,0,Grade 2,27.28525,86.526839
4,120101000051,12,1207,120703,1,30,308,9,Flat,Other,...,0,0,0,0,0,0,0,Grade 1,27.28525,86.526839
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
762101,366709001241,36,3603,360302,2,60,165,18,Flat,Mud mortar-Stone/Brick,...,0,0,0,0,0,0,0,Grade 5,28.19597,84.731540
762102,366709001251,36,3603,360302,2,35,342,18,Flat,Mud mortar-Stone/Brick,...,0,0,0,0,0,0,0,Grade 5,28.19597,84.731540
762103,366709001261,36,3603,360302,2,35,342,18,Flat,Mud mortar-Stone/Brick,...,0,0,0,0,0,0,0,Grade 5,28.19597,84.731540
762104,366709001271,36,3603,360302,2,19,306,18,Flat,Mud mortar-Stone/Brick,...,0,0,0,0,0,0,0,Grade 5,28.19597,84.731540


In [5]:
df_final

,building_id,district_id,vdcmun_id,ward_id,count_floors_pre_eq,age_building,plinth_area_sq_ft,height_ft_pre_eq,land_surface_condition,foundation_type,...,has_superstructure_stone_flag,has_superstructure_cement_mortar_stone,has_superstructure_mud_mortar_brick,has_superstructure_cement_mortar_brick,has_superstructure_timber,has_superstructure_bamboo,has_superstructure_rc_non_engineered,has_superstructure_rc_engineered,has_superstructure_other,damage_grade
0,120101000011,12,1207,120703,1,9,288,9,Flat,Other,...,0,0,0,0,0,1,0,0,0,Grade 3
1,120101000021,12,1207,120703,1,15,364,9,Flat,Other,...,0,0,0,0,0,1,0,0,0,Grade 5
2,120101000031,12,1207,120703,1,20,384,9,Flat,Other,...,0,0,0,0,0,0,0,0,0,Grade 2
3,120101000041,12,1207,120703,1,20,312,9,Flat,Other,...,0,0,0,0,0,0,0,0,0,Grade 2
4,120101000051,12,1207,120703,1,30,308,9,Flat,Other,...,0,0,0,0,0,0,0,0,0,Grade 1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
762101,366709001241,36,3603,360302,2,60,165,18,Flat,Mud mortar-Stone/Brick,...,0,0,0,0,0,0,0,0,0,Grade 5
762102,366709001251,36,3603,360302,2,35,342,18,Flat,Mud mortar-Stone/Brick,...,0,0,0,0,0,0,0,0,0,Grade 5
762103,366709001261,36,3603,360302,2,35,342,18,Flat,Mud mortar-Stone/Brick,...,0,0,0,0,0,0,0,0,0,Grade 5
762104,366709001271,36,3603,360302,2,19,306,18,Flat,Mud mortar-Stone/Brick,...,0,0,0,0,0,0,0,0,0,Grade 5


In [14]:
EPICENTER = (28.231, 84.731)


def get_geodesic_distance(row):
    building_coords = (row['latitude'], row['longitude'])
    return geodesic(building_coords, EPICENTER).km

df_final['epicenter_distance'] = df_merged.apply(get_geodesic_distance, axis=1)

df_final.dropna(inplace=True)

df_final['damage_grade'] = df_final['damage_grade'].map(lambda x: str(x[-1]) if isinstance(x, str) else x)
df_final['damage_grade'] = df_final['damage_grade'].astype('int')

df_final

,building_id,district_id,vdcmun_id,ward_id,count_floors_pre_eq,age_building,plinth_area_sq_ft,height_ft_pre_eq,land_surface_condition,foundation_type,...,has_superstructure_mud_mortar_brick,has_superstructure_cement_mortar_brick,has_superstructure_timber,has_superstructure_bamboo,has_superstructure_rc_non_engineered,has_superstructure_rc_engineered,has_superstructure_other,damage_grade,damage_grade_series,epicenter_distance
0,120101000011,12,1207,120703,1,9,288,9,Flat,Other,...,0,0,0,1,0,0,0,3,205.726354,205.726354
1,120101000021,12,1207,120703,1,15,364,9,Flat,Other,...,0,0,0,1,0,0,0,5,205.726354,205.726354
2,120101000031,12,1207,120703,1,20,384,9,Flat,Other,...,0,0,0,0,0,0,0,2,205.726354,205.726354
3,120101000041,12,1207,120703,1,20,312,9,Flat,Other,...,0,0,0,0,0,0,0,2,205.726354,205.726354
4,120101000051,12,1207,120703,1,30,308,9,Flat,Other,...,0,0,0,0,0,0,0,1,205.726354,205.726354
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
762101,366709001241,36,3603,360302,2,60,165,18,Flat,Mud mortar-Stone/Brick,...,0,0,0,0,0,0,0,5,3.882542,3.882542
762102,366709001251,36,3603,360302,2,35,342,18,Flat,Mud mortar-Stone/Brick,...,0,0,0,0,0,0,0,5,3.882542,3.882542
762103,366709001261,36,3603,360302,2,35,342,18,Flat,Mud mortar-Stone/Brick,...,0,0,0,0,0,0,0,5,3.882542,3.882542
762104,366709001271,36,3603,360302,2,19,306,18,Flat,Mud mortar-Stone/Brick,...,0,0,0,0,0,0,0,5,3.882542,3.882542


In [28]:
# Create a copy to work on
df_clean = df_final.copy()

# 1. Define the Expected Damage trend (Hazard Expectation)
# We use a log function to model how shaking intensity drops off with distance
# Buildings closer to the epicenter are expected to have higher damage
expected_damage = np.log1p(df_clean['epicenter_distance'])

# 2. Calculate the Performance Gap (Delta)
# Positive values = performed worse than expected (vulnerable)
# Negative values = performed better than expected (resilient)
damage_norm = (df_clean['damage_grade']) / 4.0
performance_gap = damage_norm - (expected_damage / expected_damage.max())

# 3. Convert the gap into a 0-100 Resilience Score
# A building that performs 'better than average' gets a high score
# We use a simple linear transformation and clip to keep values strictly 0-100
df_clean['resilience_score'] = (1.0 - performance_gap) * 50.0
df_clean['resilience_score'] = np.clip(df_clean['resilience_score'], 0.0, 100.0)

# Drop unnecessary columns
cols_to_drop = ["damage_grade", 'building_id', 'epicenter_distance', 
                'district_id', 'vdcmun_id', 'ward_id', 'damage_grade_series']
df_clean = df_clean.drop(columns=cols_to_drop)

# 1. Take the resilience score you just calculated
raw_scores = df_clean['resilience_score']

# 2. Force it to fit the 0-100 range perfectly
# This stretches the data so the lowest value becomes 0 and the highest becomes 100
min_val = raw_scores.min()
max_val = raw_scores.max()

df_clean['resilience_score'] = ((raw_scores - min_val) / (max_val - min_val)) * 100.0

# Display the resulting dataframe
print(df_clean.head())

   count_floors_pre_eq  age_building  plinth_area_sq_ft  height_ft_pre_eq  \
0                    1             9                288                 9   
1                    1            15                364                 9   
2                    1            20                384                 9   
3                    1            20                312                 9   
4                    1            30                308                 9   

  land_surface_condition foundation_type                 roof_type  \
0                   Flat           Other  Bamboo/Timber-Light roof   
1                   Flat           Other  Bamboo/Timber-Light roof   
2                   Flat           Other  Bamboo/Timber-Light roof   
3                   Flat           Other  Bamboo/Timber-Light roof   
4                   Flat           Other  Bamboo/Timber-Light roof   

  ground_floor_type other_floor_type      position  ...  \
0               Mud   Not applicable  Not attached  ...  

In [29]:
df_clean.to_csv('resilience_dataset_final.csv', index=False)
print("File saved successfully.")

File saved successfully.


In [17]:
for i in range(0,int(df_final["damage_grade"].max()),1):
    print(
        f"{i} to {i+1}: {df_final["damage_grade"][(df_final["damage_grade"]>i)\
                                         & (df_final["damage_grade"]<=i+1)].count()} rows"
        )

0 to 1: 78726 rows
1 to 2: 87019 rows
2 to 3: 136087 rows
3 to 4: 183516 rows
4 to 5: 274650 rows


In [18]:
for i in range(0,int(df_final["epicenter_distance"].max()),5):
    print(
        f"{i}km to {i+5}km: {df_final["epicenter_distance"][(df_final["epicenter_distance"]>i)\
                                         & (df_final["epicenter_distance"]<=i+5)].count()} rows"
        )

0km to 5km: 1485 rows
5km to 10km: 3373 rows
10km to 15km: 7268 rows
15km to 20km: 10989 rows
20km to 25km: 17058 rows
25km to 30km: 24483 rows
30km to 35km: 21429 rows
35km to 40km: 26428 rows
40km to 45km: 17840 rows
45km to 50km: 19410 rows
50km to 55km: 28128 rows
55km to 60km: 22672 rows
60km to 65km: 19006 rows
65km to 70km: 24114 rows
70km to 75km: 22233 rows
75km to 80km: 13553 rows
80km to 85km: 9923 rows
85km to 90km: 16615 rows
90km to 95km: 22989 rows
95km to 100km: 33383 rows
100km to 105km: 35201 rows
105km to 110km: 29847 rows
110km to 115km: 30428 rows
115km to 120km: 22313 rows
120km to 125km: 20072 rows
125km to 130km: 21648 rows
130km to 135km: 17222 rows
135km to 140km: 19272 rows
140km to 145km: 17413 rows
145km to 150km: 24078 rows
150km to 155km: 19873 rows
155km to 160km: 17052 rows
160km to 165km: 22393 rows
165km to 170km: 12470 rows
170km to 175km: 13995 rows
175km to 180km: 13554 rows
180km to 185km: 7385 rows
185km to 190km: 11959 rows
190km to 195km: 7298 

In [22]:
for i in range(0,100,5):
    print(
        f"{i}% to {i+5}%: {df_clean['resilience_score'][(df_clean['resilience_score']>i)\
                                         & (df_clean['resilience_score']<=i+5)].count()} rows"
        )

0% to 5%: 969 rows
5% to 10%: 2071 rows
10% to 15%: 5459 rows
15% to 20%: 10190 rows
20% to 25%: 24437 rows
25% to 30%: 45015 rows
30% to 35%: 70917 rows
35% to 40%: 135310 rows
40% to 45%: 49166 rows
45% to 50%: 39277 rows
50% to 55%: 85218 rows
55% to 60%: 32460 rows
60% to 65%: 34763 rows
65% to 70%: 60616 rows
70% to 75%: 21839 rows
75% to 80%: 29764 rows
80% to 85%: 36030 rows
85% to 90%: 16446 rows
90% to 95%: 35488 rows
95% to 100%: 24145 rows


In [23]:
df_final["epicenter_distance"].corr(df_clean['resilience_score'], method= 'spearman')

np.float64(0.46314171909840696)